In [ ]:
import pandas as pd
import requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/RDV_Project/processed_data/taxi_cleaned.parquet"

df = pd.read_parquet(file_path)

print(df.shape)
df.head()

(22012724, 11)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month,trip_duration,pickup_hour,time_category,trip_category
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01,8.350000,0,Malam,Pendek
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01,2.550000,0,Malam,Pendek
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01,1.950000,0,Malam,Pendek
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01,5.566667,0,Malam,Pendek
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01,3.533333,0,Malam,Pendek


In [ ]:
df["pickup_date"] = (
    pd.to_datetime(
        df["tpep_pickup_datetime"]
    ).dt.date
)

df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month,trip_duration,pickup_hour,time_category,trip_category,pickup_date
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01,8.350000,0,Malam,Pendek,2025-01-01
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01,2.550000,0,Malam,Pendek,2025-01-01
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01,1.950000,0,Malam,Pendek,2025-01-01
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01,5.566667,0,Malam,Pendek,2025-01-01
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01,3.533333,0,Malam,Pendek,2025-01-01


In [ ]:
url = (
    "https://archive-api.open-meteo.com/v1/archive"
)

params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2025-01-01",
    "end_date": "2025-06-30",
    "daily": (
        "temperature_2m_mean,"
        "precipitation_sum,"
        "rain_sum"
    ),
    "timezone": "America/New_York"
}

response = requests.get(
    url,
    params=params
)

weather_json = response.json()

In [ ]:
weather_df = pd.DataFrame({
    "pickup_date":
        weather_json["daily"]["time"],

    "avg_temperature":
        weather_json["daily"]
        ["temperature_2m_mean"],

    "precipitation":
        weather_json["daily"]
        ["precipitation_sum"],

    "rain":
        weather_json["daily"]
        ["rain_sum"]
})

In [ ]:
weather_df["pickup_date"] = pd.to_datetime(
    weather_df["pickup_date"]
).dt.date

weather_df.head()

,pickup_date,avg_temperature,precipitation,rain
0,2025-01-01,7.4,4.5,4.5
1,2025-01-02,2.6,0.0,0.0
2,2025-01-03,0.4,0.0,0.0
3,2025-01-04,-1.4,0.0,0.0
4,2025-01-05,-2.2,0.0,0.0


In [ ]:
def weather_category(rain):
    if rain == 0:
        return "Tidak Hujan"
    elif rain < 5:
        return "Hujan Ringan"
    else:
        return "Hujan Lebat"

weather_df["weather_condition"] = (
    weather_df["rain"]
    .apply(weather_category)
)

In [ ]:
merged_df = df.merge(
    weather_df,
    on="pickup_date",
    how="left"
)

print(merged_df.shape)

merged_df.head()

(22012724, 16)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month,trip_duration,pickup_hour,time_category,trip_category,pickup_date,avg_temperature,precipitation,rain,weather_condition
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01,8.350000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01,2.550000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01,1.950000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01,5.566667,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01,3.533333,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan


In [ ]:
merged_df[
    [
        "pickup_date",
        "avg_temperature",
        "rain",
        "weather_condition"
    ]
].head()

,pickup_date,avg_temperature,rain,weather_condition
0,2025-01-01,7.4,4.5,Hujan Ringan
1,2025-01-01,7.4,4.5,Hujan Ringan
2,2025-01-01,7.4,4.5,Hujan Ringan
3,2025-01-01,7.4,4.5,Hujan Ringan
4,2025-01-01,7.4,4.5,Hujan Ringan


In [ ]:
save_path = (
    "/content/drive/MyDrive/"
    "RDV_Project/processed_data/"
    "taxi_weather.parquet"
)

merged_df.to_parquet(
    save_path,
    index=False
)

print("Taxi + weather saved")

Taxi + weather saved
